# 1. Data and retrieval

## tl;dr
The executed full-document BM25 baseline reached Recall@5 **0.516667** over **200 queries / 6,006 skills**. This is retrieval, not skill execution. Saved evidence is reused with checksum validation; shared functions own computation.

## Context & Methods

Fixed seed 42 and immutable SkillRet revision. Full-document BM25 follows the pinned CPU source path; capped-metadata BM25/BGE form a separate matched comparison.

### Key assumptions
Synthetic queries, public skills, incomplete production validity. Metadata cap: 1,024 characters then 128 BGE tokens; seven vector queries exceed 512 encoder tokens. No prompts are tuned on the reserved final IDs.

In [ ]:
from pathlib import Path
from pilot import ROOT, load_data, inspect_data, baseline, read_json, rows
from trial import vectors
import pandas as pd

## Data
### 1. Verify real inputs and disjoint reservations

In [ ]:
summary = inspect_data()
assert set(read_json('manifests/pilot_query_ids.json')).isdisjoint(read_json('manifests/final_query_ids.json'))
{'revision': summary['revision'], 'counts': summary['counts'], 'integrity': summary['checks']}

### 2. Inspect consumed pilot examples only

In [ ]:
examples = read_json('results/pilot_examples.json')
pd.DataFrame([{'query_id': x['query']['id'], 'canonical_query': x['query']['query'], 'positives_evaluator_only': len(x['query']['skill_ids'])} for x in examples])

## Results
### 3. Real BM25 number, with saved raw rankings

In [ ]:
b = baseline(200)
raw = [r for r in rows('results/baseline_rows.jsonl') if r['run_id'] == b['run_id']]
independent = sum(len({s for s, _ in r['ranking'][:5]} & set(r['evaluator_only_gold'])) / len(r['evaluator_only_gold']) for r in raw) / len(raw)
assert abs(independent - b['metrics']['recall_at_5']) < 1e-12
{'queries': len(raw), 'corpus_size': b['corpus_size'], 'recall_at_5': independent, 'recall_at_10': b['metrics']['recall_at_10']}

### 4. Matched capped-metadata retrieval

In [ ]:
embedding = vectors()
pd.DataFrame([{'method': name, 'queries': r['query_count'], **r['metrics']} for name, r in [('BM25 metadata', rows('results/metadata_bm25_summary.jsonl')[-1]), ('BGE-small metadata', rows('results/vector_baseline_summary.jsonl')[-1])]])

## Takeaways
The pinned data and CPU retrieval path work. Matched-metadata BM25 Recall@5 is 0.605833 versus BGE-small 0.492500 in this exploratory sample. This does not establish a general winner. Preserve the 4,192 reserved final IDs. See reports/TRIAL_REPORT.md for limitations and distinct protocols.